# **Projeto de Graduação em Computação II [2026-Q1]**

Universidade Federal do ABC

Orientador Carlos da Silva dos Santos

### **EQUIPE**

---

Caio Cardoso Dos Santos - RA: 11202021632

Victor Ravazio de Lima - RA: 11201920941

### **IMPLEMENTAÇÃO**
---

In [24]:
# modulo para importar dataset a partir de um link
!pip install gdown

In [25]:
#Bibliotecas
import pandas as pd
import numpy as np
from google.colab import files
import gdown
import matplotlib.pyplot as plt
import random
import seaborn as sns

In [26]:
#Variaveis de controle
anotado = 400
# num_replys_anotados = 150
# num_quotes_anotados = 150
# num_originals_anotados = 150

In [27]:
# baixando datasets para o collab (não anotados)
import gdown

file_class_aborto = "1EVf978UXxIrpWCGzmDAONeXzH6_wwbCM"
file_class_cloroquina = "1ofZUDIRto5VcdCVt-mIt3f4nMSo-byzJ"
file_class_contas_suspensas = "1PeM1gVXV4PVj0tR3ausqvjD-tj_R-G0U"
file_class_mandetta = "1xqzi-pqhDsMmSpmvp0upOznqlHwkhvmd"

url_1 = f"https://drive.google.com/uc?id={file_class_aborto}"
url_2 = f"https://drive.google.com/uc?id={file_class_cloroquina}"
url_3 = f"https://drive.google.com/uc?id={file_class_contas_suspensas}"
url_4 = f"https://drive.google.com/uc?id={file_class_mandetta}"


gdown.download(url_1, "aborto.csv", quiet=False)
gdown.download(url_2, "cloroquina.csv", quiet=False)
gdown.download(url_3, "contas_suspensas.csv", quiet=False)
gdown.download(url_4, "mandetta.csv", quiet=False)

Downloading...
From: https://drive.google.com/uc?id=1EVf978UXxIrpWCGzmDAONeXzH6_wwbCM
To: /content/aborto.csv
100%|██████████| 4.22M/4.22M [00:00<00:00, 22.8MB/s]
Downloading...
From: https://drive.google.com/uc?id=1ofZUDIRto5VcdCVt-mIt3f4nMSo-byzJ
To: /content/cloroquina.csv
100%|██████████| 12.2M/12.2M [00:00<00:00, 49.4MB/s]
Downloading...
From: https://drive.google.com/uc?id=1PeM1gVXV4PVj0tR3ausqvjD-tj_R-G0U
To: /content/contas_suspensas.csv
100%|██████████| 9.94M/9.94M [00:00<00:00, 23.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1xqzi-pqhDsMmSpmvp0upOznqlHwkhvmd
To: /content/mandetta.csv
100%|██████████| 10.9M/10.9M [00:00<00:00, 155MB/s]


'mandetta.csv'

In [28]:
#carregando o dataset os datasets tematicos
df_aborto = pd.read_csv ("aborto.csv")
df_cloroquina = pd.read_csv ("cloroquina.csv")
df_contas_suspensas = pd.read_csv ("contas_suspensas.csv")
df_mandetta = pd.read_csv ("mandetta.csv")

In [29]:
#numero total de instancias no dataset
(df_aborto.shape, df_cloroquina.shape, df_contas_suspensas.shape, df_mandetta.shape)

((13849, 6), (42507, 6), (30265, 6), (36029, 6))

In [30]:
def cria_dataset_exploratorio(
    df,
    prob_col="P(Conflict)",
    frac_random=1/3,
    faixa_incerta=(0.40, 0.60),
    faixa_alta=(0.80, 0.90),
    n_incerta=50,
    n_alta=25,
    n_random=25,
    random_state=42
):
    """
    Cria um dataset para anotação seguindo a regra:
    - n_incerta exemplos da faixa_incerta
    - n_alta exemplos da faixa_alta
    - n_random exemplos aleatórios

    O ciclo é repetido até um dos conjuntos acabar.
    """


    # 1) Seleciona 1/3 aleatório
    df_random = df.sample(frac=frac_random, random_state=random_state)
    df_restante = df.drop(df_random.index)
    df_random['Tipo'] = 'Random'

    # 2) Região incerta

    df_incerta = df_restante[df_restante[prob_col].between(faixa_incerta[0], faixa_incerta[1])]

    df_restante = df_restante.drop(df_incerta.index)
    df_incerta = df_incerta.sample(frac=1, random_state=random_state)
    df_incerta['Tipo'] = 'Incerta'

    # 3) Região alta

    df_alta = df_restante[df_restante[prob_col].between(faixa_alta[0],faixa_alta[1])]
    df_restante = df_restante.drop(df_alta.index)
    df_alta = df_alta.sample(frac=1, random_state=random_state)
    df_alta['Tipo'] = 'Alta'

    # 4) Calculando o numero de blocos (minimo do qtde do numero de blocos que podemos construir com cade df)

    n_blocos = min(len(df_incerta) // n_incerta, len(df_alta) // n_alta, len(df_random) // n_random)


    # 5) Montando o dataset final

    blocos = []

    for i in range(n_blocos):

        bloco = pd.concat([df_incerta.iloc[i*n_incerta:(i+1)*n_incerta], df_alta.iloc[i*n_alta:(i+1)*n_alta],df_random.iloc[i*n_random:(i+1)*n_random]])

        blocos.append(bloco)

    df_final = pd.concat(blocos, ignore_index=True)

    return df_final

In [31]:
#numero total de instancias no dataset
df_exploratorio_aborto = cria_dataset_exploratorio(df_aborto)
df_exploratorio_cloroquina = cria_dataset_exploratorio(df_cloroquina)
df_exploratorio_contas_suspensas = cria_dataset_exploratorio(df_contas_suspensas)
df_exploratorio_mandetta = cria_dataset_exploratorio(df_mandetta)

In [34]:
(df_exploratorio_aborto.shape, df_exploratorio_cloroquina.shape, df_exploratorio_contas_suspensas.shape, df_exploratorio_mandetta.shape)

((1100, 7), (2100, 7), (1300, 7), (1500, 7))

In [32]:
df_exploratorio_mandetta.iloc[150:200,:]

,tweet_1,tweet_2,Class,P(NoConflict),P(Conflict),Conflict,Tipo
150,Mandetta foi escolhido Ministro com a missão d...,acho ingênuo achar que Mandetta virou pro-SUS ...,2,0.170198,0.829802,NaN,Alta
151,"Em sua despedida, Mandetta defende a vida, o S...",O SUS? RAPAZ,2,0.199062,0.800938,NaN,Alta
152,"Em sua despedida, Mandetta defende a vida, o S...",Compatível mesmo é o rombo bilionário que o PT...,2,0.109780,0.890220,NaN,Alta
153,"Após a saída do Mandetta, quem deveria ser o p...","deixe nosso presidente governar, vc é um bosta...",2,0.185385,0.814615,NaN,Alta
154,"Em sua despedida, Mandetta defende a vida, o S...",Inacreditável que o Haddad romantiza o verdade...,2,0.150742,0.849258,NaN,Alta
155,"Bolsonaro demitiu Mandetta por ciúme, inveja e...","Como os caras conseguem colocar ""ciúme""?? A VI...",2,0.122592,0.877408,NaN,Alta
156,"Em sua despedida, Mandetta defende a vida, o S...","A saúde é vida, mas sem trabalhar para se sust...",2,0.104527,0.895473,NaN,Alta
157,"""E o Mandetta, a linha dele, como médico, era ...",que preocupação tola para um médico .....,2,0.108666,0.891334,NaN,Alta
158,"""E o Mandetta, a linha dele, como médico, era ...",talvez pq ele fosse o ministro da SAÚDE????,2,0.109758,0.890242,NaN,Alta
159,"""E o Mandetta, a linha dele, como médico, era ...",Meu dindo smp diz: “Vc não é super heroína. Se...,2,0.196338,0.803662,NaN,Alta


In [33]:
df_exploratorio_aborto.to_csv('aborto_anotar_exploratorio.csv', index=False)
files.download('aborto_anotar_exploratorio.csv')
df_exploratorio_cloroquina.to_csv('cloroquina_anotar_exploratorio.csv', index=False)
files.download('cloroquina_anotar_exploratorio.csv')
df_exploratorio_contas_suspensas.to_csv('contas_suspensas_anotar_exploratorio.csv', index=False)
files.download('contas_suspensas_anotar_exploratorio.csv')
df_exploratorio_mandetta.to_csv('mandetta_anotar_exploratorio.csv', index=False)
files.download('mandetta_anotar_exploratorio.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>